# Phase 5 — New Baselines (Reviewer Comment 1)
Adds Boruta, RFE, and mRMR as additional comparison methods.

In [1]:
import pandas as pd
import numpy as np
import shap
import time
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from itertools import combinations
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
from scipy.stats import spearmanr
from boruta import BorutaPy
import mrmr
print('All imports successful')

e:\project1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful


In [2]:
BASE    = Path('E:/project1/EW_MOGA_Project')
DATA    = BASE / 'data' / 'Brain Tumor.csv'
RESULTS = BASE / 'results'
RESULTS.mkdir(exist_ok=True)

FEATURE_COLS = ['Mean','Variance','Standard Deviation','Entropy',
    'Skewness','Kurtosis','Contrast','Energy',
    'ASM','Homogeneity','Dissimilarity','Correlation','Coarseness']

N_FEATURES = 13
N_FOLDS    = 5
SEEDS      = [0, 1, 2]
N_PERTURB  = 5
N_SELECT   = 6   # number of features to select (same as SHAP-RF)

print('Configuration set')

Configuration set


In [3]:
df    = pd.read_csv(DATA)
X_raw = df[FEATURE_COLS].values.astype(float)
y_raw = df['Class'].values.astype(int)
print(f'Dataset: {X_raw.shape}')

def preprocess(X_tr, X_val, y_tr, random_state=42):
    scaler   = MinMaxScaler()
    X_tr_sc  = scaler.fit_transform(X_tr)
    X_val_sc = scaler.transform(X_val)
    sm = SMOTE(random_state=random_state)
    X_tr_sm, y_tr_sm = sm.fit_resample(X_tr_sc, y_tr)
    return X_tr_sm, X_val_sc, y_tr_sm

print('Preprocessing defined')

Dataset: (3762, 13)
Preprocessing defined


In [4]:
# ── Audit functions ──────────────────────────────────────────
def compute_ccc(X_tr, y_tr, X_val, idx, seed=42):
    import xgboost as xgb
    from sklearn.svm import SVC
    from sklearn.neural_network import MLPClassifier
    classifiers = {
        'RF' : RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1),
        'XGB': xgb.XGBClassifier(n_estimators=100, random_state=seed,
                                  eval_metric='logloss', verbosity=0),
        'SVM': SVC(kernel='rbf', probability=True, random_state=seed),
        'MLP': MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=seed),
    }
    rankings = []
    rng = np.random.RandomState(seed)
    for name, clf in classifiers.items():
        try:
            clf.fit(X_tr[:, idx], y_tr)
            if name in ['RF','XGB']: exp = shap.TreeExplainer(clf)
            else:
                bg = X_tr[:, idx][rng.choice(len(X_tr), min(50,len(X_tr)), replace=False)]
                exp = shap.KernelExplainer(clf.predict_proba, bg)
            sv = exp.shap_values(X_val[:min(30,len(X_val))][:, idx], check_additivity=False)
            if isinstance(sv, list): sv_arr = np.mean([np.abs(r) for r in sv], axis=0)
            else: sv_arr = np.abs(sv)
            if sv_arr.ndim == 3: sv_arr = sv_arr.mean(axis=2)
            rankings.append(sv_arr.mean(axis=0))
        except: pass
    if len(rankings) < 2: return 0.0
    rhos = [spearmanr(rankings[i], rankings[j])[0]
            for i,j in combinations(range(len(rankings)),2)]
    return float(np.mean([r if not np.isnan(r) else 0.0 for r in rhos]))

def compute_pr(X_tr, y_tr, idx, seed=42):
    rf_b = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)
    rf_b.fit(X_tr[:, idx], y_tr)
    rng = np.random.RandomState(seed)
    bg  = X_tr[rng.choice(len(X_tr), min(50,len(X_tr)), replace=False)]
    exp = shap.TreeExplainer(rf_b, bg)
    sv  = exp.shap_values(X_tr[:min(30,len(X_tr)),:], check_additivity=False)
    if isinstance(sv, list): sv_arr = np.mean([np.abs(r) for r in sv], axis=0)
    else: sv_arr = np.abs(sv)
    if sv_arr.ndim == 3: sv_arr = sv_arr.mean(axis=2)
    base_rank = sv_arr.mean(axis=0)
    rhos = []
    for p in range(N_PERTURB):
        boot_idx = rng.choice(len(X_tr), len(X_tr), replace=True)
        X_b, y_b = X_tr[boot_idx], y_tr[boot_idx]
        rf_p = RandomForestClassifier(n_estimators=100, random_state=seed+p, n_jobs=-1)
        rf_p.fit(X_b[:, idx], y_b)
        bg_p  = X_b[rng.choice(len(X_b), min(50,len(X_b)), replace=False)]
        exp_p = shap.TreeExplainer(rf_p, bg_p)
        sv_p  = exp_p.shap_values(X_b[:min(30,len(X_b)),:], check_additivity=False)
        if isinstance(sv_p, list): sv_p_arr = np.mean([np.abs(r) for r in sv_p], axis=0)
        else: sv_p_arr = np.abs(sv_p)
        if sv_p_arr.ndim == 3: sv_p_arr = sv_p_arr.mean(axis=2)
        rho, _ = spearmanr(base_rank, sv_p_arr.mean(axis=0))
        rhos.append(rho if not np.isnan(rho) else 0.0)
    return float(np.mean(rhos))

print('Audit functions defined')

Audit functions defined


In [5]:
# ── Boruta ───────────────────────────────────────────────────
def run_boruta(X_tr, y_tr, X_val, y_val, seed=42):
    rf  = RandomForestClassifier(n_jobs=-1, random_state=seed)
    bor = BorutaPy(rf, n_estimators='auto', verbose=0, random_state=seed)
    bor.fit(X_tr, y_tr)
    idx = list(np.where(bor.support_)[0])
    if not idx:
        idx = list(np.where(bor.support_weak_)[0])
    if not idx:
        idx = list(range(N_SELECT))
    rf_f = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)
    rf_f.fit(X_tr[:, idx], y_tr)
    acc = accuracy_score(y_val, rf_f.predict(X_val[:, idx]))
    ccc = compute_ccc(X_tr, y_tr, X_val, idx, seed)
    pr  = compute_pr(X_tr, y_tr, idx, seed)
    return acc, len(idx), ', '.join([FEATURE_COLS[i] for i in idx]), ccc, pr

# ── RFE ──────────────────────────────────────────────────────
def run_rfe(X_tr, y_tr, X_val, y_val, seed=42):
    rf  = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)
    rfe = RFE(estimator=rf, n_features_to_select=N_SELECT, step=1)
    rfe.fit(X_tr, y_tr)
    idx = list(np.where(rfe.support_)[0])
    rf_f = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)
    rf_f.fit(X_tr[:, idx], y_tr)
    acc = accuracy_score(y_val, rf_f.predict(X_val[:, idx]))
    ccc = compute_ccc(X_tr, y_tr, X_val, idx, seed)
    pr  = compute_pr(X_tr, y_tr, idx, seed)
    return acc, len(idx), ', '.join([FEATURE_COLS[i] for i in idx]), ccc, pr

# ── mRMR ─────────────────────────────────────────────────────
def run_mrmr(X_tr, y_tr, X_val, y_val, seed=42):
    X_df = pd.DataFrame(X_tr, columns=FEATURE_COLS)
    y_s  = pd.Series(y_tr)
    selected = mrmr.mrmr_classif(X=X_df, y=y_s, K=N_SELECT)
    idx = [FEATURE_COLS.index(f) for f in selected if f in FEATURE_COLS]
    if not idx: idx = list(range(N_SELECT))
    rf_f = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)
    rf_f.fit(X_tr[:, idx], y_tr)
    acc = accuracy_score(y_val, rf_f.predict(X_val[:, idx]))
    ccc = compute_ccc(X_tr, y_tr, X_val, idx, seed)
    pr  = compute_pr(X_tr, y_tr, idx, seed)
    return acc, len(idx), ', '.join([FEATURE_COLS[i] for i in idx]), ccc, pr

print('Boruta, RFE, mRMR functions defined')

Boruta, RFE, mRMR functions defined


In [6]:
# ── Run all new baselines with checkpoint saving ──────────────
CHECKPOINT_FILE_NEW = RESULTS / 'new_baselines_checkpoint.csv'

# Resume from checkpoint if exists
if CHECKPOINT_FILE_NEW.exists():
    df_existing = pd.read_csv(CHECKPOINT_FILE_NEW)
    results     = df_existing.to_dict('records')
    completed   = set(zip(df_existing['method'],
                          df_existing['seed'],
                          df_existing['fold']))
    print(f'Resuming from checkpoint — {len(results)} runs already done')
    print(f'Completed: {df_existing.method.value_counts().to_dict()}')
else:
    results   = []
    completed = set()
    print('Starting fresh — no checkpoint found')

new_methods = {
    'Boruta': run_boruta,
    'RFE'   : run_rfe,
    'mRMR'  : run_mrmr,
}

total_runs = len(new_methods) * len(SEEDS) * N_FOLDS
run_num    = len(results)

for method_name, method_fn in new_methods.items():
    print(f'\n--- {method_name} ---')
    for seed in SEEDS:
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_raw, y_raw)):

            # Skip already completed runs
            if (method_name, seed, fold) in completed:
                print(f'  [SKIP] {method_name} seed={seed} fold={fold} already done')
                continue

            run_num += 1
            print(f'  [{run_num}/{total_runs}] seed={seed} fold={fold}', end=' ... ')
            t0 = time.time()

            X_tr,  X_val  = X_raw[tr_idx],  X_raw[val_idx]
            y_tr,  y_val  = y_raw[tr_idx],  y_raw[val_idx]
            X_tr_pp, X_val_pp, y_tr_pp = preprocess(X_tr, X_val, y_tr, seed)

            acc, n_feat, feat_names, ccc, pr = method_fn(
                X_tr_pp, y_tr_pp, X_val_pp, y_val, seed)
            elapsed = time.time() - t0

            results.append({
                'method'               : method_name,
                'seed'                 : seed,
                'fold'                 : fold,
                'accuracy'             : acc,
                'n_features'           : n_feat,
                'runtime_sec'          : elapsed,
                'feature_names'        : feat_names,
                'in_loop_explainability': np.nan,
                'audit_ccc'            : ccc,
                'audit_pr'             : pr,
            })

            # Save checkpoint after every single run
            pd.DataFrame(results).to_csv(CHECKPOINT_FILE_NEW, index=False)
            print(f'acc={acc:.4f} ccc={ccc:.3f} pr={pr:.3f} [{elapsed:.0f}s] ✓ saved')
            completed.add((method_name, seed, fold))

df_new = pd.DataFrame(results)
df_new.to_csv(RESULTS / 'new_baselines_results.csv', index=False)
print(f'\nAll {len(df_new)} runs complete')
print(f'Saved: results/new_baselines_results.csv')

Starting fresh — no checkpoint found

--- Boruta ---
  [1/45] seed=0 fold=0 ... 

100%|██████████| 30/30 [00:01<00:00, 18.88it/s]


acc=0.9867 ccc=0.458 pr=0.963 [44s] ✓ saved
  [2/45] seed=0 fold=1 ... 

100%|██████████| 30/30 [00:03<00:00,  8.57it/s]


acc=0.9907 ccc=0.450 pr=0.935 [69s] ✓ saved
  [3/45] seed=0 fold=2 ... 

100%|██████████| 30/30 [00:03<00:00,  9.44it/s]


... (81 outputs left out)

100%|██████████| 30/30 [00:00<00:00, 146.19it/s]


acc=0.9787 ccc=0.800 pr=0.991 [10s] ✓ saved
  [45/45] seed=2 fold=4 ... 

100%|██████████| 30/30 [00:00<00:00, 147.94it/s]


acc=0.9747 ccc=0.562 pr=0.997 [10s] ✓ saved

All 45 runs complete
Saved: results/new_baselines_results.csv


In [7]:
out_path = RESULTS / 'new_baselines_results.csv'
df_new.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print()
for method in df_new['method'].unique():
    sub = df_new[df_new['method']==method]
    print(f'{method}:')
    print(f'  Accuracy : {sub.accuracy.mean():.4f} ± {sub.accuracy.std():.4f}')
    print(f'  CCC      : {sub.audit_ccc.mean():.4f} ± {sub.audit_ccc.std():.4f}')
    print(f'  PR       : {sub.audit_pr.mean():.4f} ± {sub.audit_pr.std():.4f}')
print()
print('Notebook 5 complete — ready to run Notebook 6')

Saved: E:\project1\EW_MOGA_Project\results\new_baselines_results.csv

Boruta:
  Accuracy : 0.9872 ± 0.0038
  CCC      : 0.4120 ± 0.0521
  PR       : 0.9438 ± 0.0203
RFE:
  Accuracy : 0.9845 ± 0.0025
  CCC      : 0.5092 ± 0.0939
  PR       : 0.9208 ± 0.0222
mRMR:
  Accuracy : 0.9829 ± 0.0040
  CCC      : 0.5790 ± 0.1223
  PR       : 0.9776 ± 0.0154

Notebook 5 complete — ready to run Notebook 6
